# Day 24: Continuous Batching 状态转移

用 arrival、token budget、running slots 约束调度。轮内不能重复执行同一请求；完成请求释放资源，下一轮可补入等待请求。先比较模拟轨迹，不换算 GPU tokens/s。

参考源码：`learning/mini_vllm/scheduler.py`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY22_30.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 24完整讲义](../docs/DAY24_SCHEDULER_FAIRNESS.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

运行名额像两张工作台，预算像这一轮允许做的工序数量：

```text
slots=2：最多两条请求留在running
budget=1：本轮只让其中一条执行一次
arrival：什么时候开始有资格排队
```

A=3、B=1在round=0到达，C=2在round=1到达；slots=2，共6个decode工作项。
budget=2时本例三种策略为6/5/3轮；
budget=1时本例三者都为6轮。连续补位不是任意条件下总轮数更少的定律。
晚到请求何时开始、长请求是否饿死，与总轮数同样重要。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(24, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

等待、公平轮转、完成三个事件如何记录？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：slots和budget相同吗？**
答：前者限制运行集合容量，后者限制本轮实际工作量。

**问：first_round=2而trace round=1矛盾吗？**
答：不矛盾，前者是完成首次执行后的计数，后者从0编号。

**问：吞吐最高一定最公平吗？**
答：不一定，大请求饥饿或短请求尾延迟可能很差，要先定义服务目标。

**问：这里能证明prefill优先或decode优先哪种更好吗？**
答：不能，本模拟没有prefill成本，只能验证所实现的decode逻辑。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY24_SCHEDULER_FAIRNESS.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。